# DS4DS Exercise Sheet 04

**General Instructions:**

- Review the weekly course material (lectures, readings, slides, etc.) before starting the exercises.
- Complete all assigned exercises independently before the Q&A session.
- Please use Julia Version 1.10.x to ensure compatibility.
- Please only write between the `#--- YOUR CODE STARTS HERE ---#` and `#--- YOUR CODE ENDS HERE ---#` comments.

### Task 1: Least Squares Linear Approximation

The measurements shown in the figure below were gathered from an unknown process.

<img src="measurements_updated.png" alt="measurements" width="400"/>

However, we can assume a linear relationship between the measurements $y$ and the inputs $u$ for the underlying process

$$
\begin{align}
    y[k] = w_0 + w_1 u[k]
\end{align}.
$$

Your task is to identify the parameter vector $$\boldsymbol{w} = \begin{bmatrix} w_0\\ w_1\end{bmatrix} $$ using the least squares approach shown in the lecture.

**Note:** You can solve this problem using either pen and paper or Julia. Place your calculated values for $w_0$ and $w_1$ into the variables below.

**Tip:** Think about the role of the two parameters you have acquired. Do they make sense for the data? Is there a way to visualize your result?

In [45]:
w_0 = NaN
w_1 = NaN

### BEGIN SOLUTION

y = [2, 2, 4]  # measurements
u = [1, 2, 3]

Z = hcat(ones(3), u)  # regressor matrix

w = inv(transpose(Z) * Z) * transpose(Z) * y

w_0 = w[1];
w_1 = w[2];

### END SOLUTION

1.0

### Task 2: Parameter Estimation of the DC Motor

The electrical behaviour of the DC motor in steady state is characterized by the following static equation

$$
\begin{align}
    i_s = u_s \frac{1}{R_s} - \omega \frac{\Psi_E}{R_s}
\end{align}
$$

where $u_s$ is the voltage provided by a power supply, $\omega$ is the rotation speed which is set by a mechanical load machine, $R_s$ and $\Psi_E$ are unknown values and $i_s$ is the current that can be measured after the motor reached steady-state operation. For the whole task, you can assume that $u_s$ and $\omega$ are perfectly known, while the measured values for $i_s$ are affected by bias-free, additive, Gaussian noise.

**a)** Consider the system described above. Assign each of the equation components to the different elements of the least squares problem. For this, map each of the components in the cell below to one of the following classes:
- class **1**: Measurements $\boldsymbol{y}$
- class **2**: Regressors $\boldsymbol{z}$
- class **3**: Parameters $\boldsymbol{w}$ (this is the letter $\boldsymbol{w}$ for the parameter vector not to be confused with the greek symbol $\omega$ for the angular velocity)

In [46]:
# choose 1,2 or 3

class_i_s = 0
class_u_s = 0
class_one_over_R_s = 0  # i.e. 1/R_s
class_omega = 0
class_minus_Psi_E_over_R_s = 0  # i.e. - Psi_E/R_s

### BEGIN SOLUTION

class_i_s = 1;
class_u_s = 2;
class_one_over_Rs = 3;
class_omega = 2;
class_minus_Psi_E_over_R_s = 3;

### END SOLUTION

3

**b)** Implement the function below to calculate the parameters $\boldsymbol{w}$ from the regressor matrix $\boldsymbol{Z}$ and the measurement vector $\boldsymbol{y}$ using the ordinary least squares approach from the lecture.

**Hint**: This is not specific to the system at hand.

In [47]:
function parameter_calculation_OLS(Z, y)
    """
    Args:
        Z: regressor matrix with shape (n_measurements, n_regressors)
        y: measurement_vector with shape (n_measurements)
    
    Returns:
        The parameter vector w with shape (n_parameters)
    """
    
    ### BEGIN SOLUTION
    w = inv(transpose(Z) * Z) * transpose(Z) * y
    
    ### END SOLUTION
    return w
end;

**c)** Use the function implemented in subtask b) and the given data for $u_s$, $\omega$ and $i_s$ to estimate values for $a = \frac{1}{R_s}$ and $b = -\frac{\Psi_E}{R_s}$. Put your results for $a$ and $b$ into the variables below. 

In [48]:
using MAT

data_task_2 = matopen("data_task_2.mat")

# read data
u_s = read(data_task_2, "u_s");
omega = read(data_task_2, "omega");
i_s = read(data_task_2, "i");

In [49]:
# variables for you to overwrite
a = NaN
b = NaN

### BEGIN SOLUTION
Z = cat(dims=2, u_s, omega)
y = i_s

w = parameter_calculation_OLS(Z, y)

a = w[1];
b = w[2];

### END SOLUTION

-0.043177661496966266

**d)** Implement a function to compute estimations for $R_s$ and $\Psi_E$ based on $a$ and $b$.

In [50]:
function compute_Rs_Psi_E(a, b)

    ### BEGIN SOLUTION
    R_s = 1 / a
    Psi_E = - R_s * b
    
    ### END SOLUTION

    return R_s, Psi_E
end;

**e)** Analyse the accuracy of your parameter estimates. For this, take into consideration that the measured values for $i_{s}$, that you loaded from the file, are affected by bias-free, additive, Gaussian noise:

$$
\begin{align}
i_{s} = i_{s, true} + \nu \quad \mathrm{with} \, \nu \sim \mathcal{N}(0, \sigma_n^2).
\end{align}
$$

i) Estimate the corrected sample variance of $\nu$ from the measurements for $i_s$, that you were given, and your estimate $\hat{i}_s$, that is based on the estimated parameters.

In [51]:
var_nu = NaN

### BEGIN SOLUTION

Z = cat(dims=2, u_s, omega)
y = i_s

w = parameter_calculation_OLS(Z, y)
y_est = Z * w

var_nu = 1 / (length(y_est) - 1) * sum((y - y_est).^2)

### END SOLUTION

2.751046876116658

ii) Implement the function below to estimate the covariance matrix of the parameter vector $\boldsymbol{w}$ based on the noise variance $\sigma_n^2$ and the regressor matrix $\boldsymbol{Z}$

In [52]:
function estimate_covariance_matrix(var_nu, Z)
    """
    Args:
        var_nu: variance of the noise process
        Z: regressor matrix with shape (n_measurements, n_regressors)

    Returns:
        covariance matrix of w
    """
    
    ### BEGIN SOLUTION

    cov_w = var_nu * inv(transpose(Z) * Z)

    ### END SOLUTION

    return cov_w
end;

In [53]:
using LinearAlgebra

iii) Evaluate the coefficient of determination for the regression problem (see https://en.wikipedia.org/wiki/Coefficient_of_determination).

$$
\begin{align}
    R^2 = 1 - \frac{\sum_i{(y_i - \hat{y}_i)^2}}{\sum_i{(y_i - \bar{y})^2}}
\end{align}
$$

In [54]:
R_squared = NaN  # put your solution into this variable

### BEGIN SOLUTION

Z = cat(dims=2, u_s, omega)
y = i_s
mean_y = sum(y) / length(y)

y_est = Z * parameter_calculation_OLS(Z, y)

numer = sum((y - y_est).^2)
denom = sum((y .- mean_y).^2)

R_squared = 1 - numer / denom

### END SOLUTION

0.9827234782875744

In [55]:
mean_y

1.4635906751159424

In [56]:
using Statistics

In [57]:
i_s_mean = mean(i_s)

1.4635906751159424

### Task 3: Estimate Polynom Coefficients

The given data was produced using a polynomial of the form


$$
\begin{align}
    y = w_0 + x \cdot w_1 + \dots + x^p \cdot w_p
\end{align}
$$

with an unknown degree $p$.

Below you are given data for the inputs $x$ and the outputs $y$. On the one hand you are given training data you will use for the identification of coefficients (estimation of a model) and validation data
that is used to evaluate the generalization performance of your model. All of the data for the measurements $y$ is affected by bias-free, additive, Gaussian noise, i.e., the values you are given below do not fully follow the formula given above, but instead

$$
\begin{align}
    y = w_0 + x \cdot w_1 + \dots + x^p \cdot w_p + \nu \quad \mathrm{with} \, \nu \sim \mathcal{N}(0, \sigma_n^2).
\end{align}
$$

The variance of the noise process is unknown, but it is also not necessary to estimate it for completing this task.

In [58]:
data_task_3 = matopen("data_task_3.mat");

In [59]:
# training data
x_training = read(data_task_3, "x_training")[1, :];
y_training = read(data_task_3, "y_training")[1, :];
 
# validation data
x_validation = read(data_task_3, "x_validation")[1, :];
y_validation = read(data_task_3, "y_validation")[1, :];

**a)** Consider the regression problem of the form:

$$
\begin{align}
    \boldsymbol{y} = \boldsymbol{Z} \cdot \boldsymbol{w}  + \boldsymbol{\nu}.
\end{align}
$$

Implement the function below to compute the regressor matrix $\boldsymbol{Z}$ for the polynom coefficient estimation problem described above. The function takes the input data $\boldsymbol{x}$ and the degree of the polynom $p$ as inputs.

**Example:** If $p=2$, we get a model until the 2nd polynomial degree

$$
\begin{align}
    y = w_0 + x \cdot w_1 + x^2 \cdot w_2
\end{align}
$$

which results in the regressor vector

$$
\begin{align}
    \boldsymbol{z} = \begin{bmatrix} 1 & x & x^2 \end{bmatrix}.
\end{align}
$$

With an input vector $\boldsymbol{x} = \begin{bmatrix} 1 & 2 & 3 \end{bmatrix}$ this results in the regressor matrix

$$
\begin{align}
    \boldsymbol{Z} = \begin{bmatrix}
        1 & 1 & 1 \\
        1 & 2 & 4 \\
        1 & 3 & 9 \\
    \end{bmatrix}.
\end{align}
$$



In [60]:
function regressor_matrix(x, p)
    """
    Args:
        x: Input data
        p: Degree of the polynomial
    
    Returns:
        Regressor matrix Z with shape (length(x), p+1)
    """
    
    ### BEGIN SOLUTION

    Z = zeros((length(x), p+1))
    for i in range(0, p)
        Z[:, i+1] = x.^i
    end

    ### END SOLUTION

    return Z
end

regressor_matrix (generic function with 1 method)

**b)** Write the computation of the parameter estimation using OLS as a function of the input data $\boldsymbol{x}$, the output data $\boldsymbol{y}$ and the degree of the polynomial $p$. Your goal is to find the parameters $\boldsymbol{w}$ for the polynom of degree $p$. Use the function for the computation of the regression matrix that was asked for in subtask a) 

In [61]:
function compute_parameter_estimation(x, y, p)
    """
    Args:
        x: Input data
        y: Output data
        p: Degree of the polynomial
    
    Returns:
        parameter vector w with shape (p+1)
    """
    
    ### BEGIN SOLUTION

    Z = regressor_matrix(x, p)
    w = inv(transpose(Z) * Z) * transpose(Z) * y
    
    ### END SOLUTION

    return w
end

compute_parameter_estimation (generic function with 1 method)

**c)** Your goal now is to find the fit onto the training data that best generalizes to the validation data.

For this:
- iterate over the degree of the polynomial $p$,
- find the parameter vector $\boldsymbol{w}_p$ based on the training data $\boldsymbol{x}_{training}$ and $\boldsymbol{y}_{training}$,
- compute the estimates $\hat{\boldsymbol{y}}$ using $\boldsymbol{w}_p$ and $\boldsymbol{x}_{validation}$, and
- compare the estimates $\hat{\boldsymbol{y}}$ with $\boldsymbol{y}_{validation}$ using the given error function.

In [62]:
function error_function(a, b)
    """Computes the error between the two input vectors."""

    @assert length(a) == length(b)
    @assert ndims(a) == ndims(b) == 1
    
    N = length(a)
    return sqrt(1/N * sum((a-b).^2))
end

error_function (generic function with 1 method)

Find a polynomial model with **an error below $10$** according to the steps described above and report its degree in the variable $p_{sol}$ below. 

**Hint 1:** $p_{sol} \in [0, 10]$

**Hint 2:** Multiple degrees for the polynomial produce an error that is low enough. 

In [63]:
p_sol = NaN

### BEGIN SOLUTION

for p in range(0, 10)
    w = compute_parameter_estimation(x_training, y_training, p)
    println(w)
    y_est = regressor_matrix(x_validation, p) * w
    error = error_function(y_est[:, 1], y_validation[:, 1])
    
    println("Current error: ", error)
    if error < 10
        p_sol = p
        break
    end
end

### END SOLUTION

[-0.6212047034029189]
Current error: 51.82270393142087
[-0.6212047034029194, -6.149024114577696]
Current error: 31.01722337812792
[-1.880671778091551, -6.149024114577696, 0.13674213953762288]
Current error: 31.05846253040979
[-1.8806717780915536, -0.18536169585629225, 0.136742139537623, -0.36091904998464847]
Current error: 7.376833189338484


 Plot the estimates of the model, the training data and the validation data for each value of $p \in [0, 10]$. Why is the highest degree of freedom not necessarily the best choice? What effect that you might know from the context of machine learning can be seen here?

In [64]:
for p in 0:10
    w_p = compute_parameter_estimation(x_training, y_training, p)
    println(w_p)
    y_hat = regressor_matrix(x_validation, p) * w_p
    err = error_function(y_hat, y_validation)
    println(err)
    if (p_sol == -1 && err<10)
        p_sol = err
    end
    #display(err)
end

[-0.6212047034029189]
51.82270393142087
[-0.6212047034029194, -6.149024114577696]
31.01722337812792
[-1.880671778091551, -6.149024114577696, 0.13674213953762288]
31.05846253040979
[-1.8806717780915536, -0.18536169585629225, 0.136742139537623, -0.36091904998464847]
7.376833189338484
[-2.081329653472189, -0.18536169585629225, 0.21022790356895438, -0.36091904998464847, -0.0031288715367679062]
7.248244351445751
[-2.0813296534722046, -3.6399296275600648, 0.21022790356896137, 0.23356353705000288, -0.003128871536768209, -0.019658767687283252]
34.608697098379494
[-1.410214500258383, -3.639929627560045, -0.3159630298509297, 0.23356353705000243, 0.05559089449668997, -0.01965876768728305, -0.001595730990035002]
38.23522411608347
[-1.4102145002580064, -0.09012507498823652, -0.3159630298512006, -0.9722240252322081, 0.05559089449671911, 0.08006760309654837, -0.001595730990035767, -0.002311474789917334]
166.8951346679973
[1.0606994909587328, -0.09012507498492543, -3.730477042336844, -0.97222402523313

In [65]:
p_sol

3